# cdc_loader — exploratory tests

Run each cell in order. All tables are written to `data/warehouse.duckdb`.

- **Bronze**: `raw_transactions` ← `src.bronze.cdc_loader`
- **Silver**: `silver_reconciliation_runs`, `silver_reconciliation_results`, `silver_enterprise_company` ← `src.silver.*`

In [18]:
import sys
sys.path.insert(0, '..')

import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

DATA = Path('..') / 'tmp_sample_data_test'
assert DATA.exists(), f"Sample data directory not found: {DATA.resolve()}"
print(f"DATA = {DATA.resolve()}")

DATA = C:\Users\Leal\data-engineer-challenge\tmp_sample_data_test


## 1 — Load bronze: transactions

In [19]:
from src.bronze.cdc_loader import load_transactions

load_transactions([
    DATA / 'transactions_batch_1.parquet',
    DATA / 'transactions_batch_2.parquet',
])

INFO  raw_transactions: 956 raw rows → 955 final rows (1 removed by dedup+deletes)


955

## 2 — Seed silver: reconciliation runs, results, and enterprise company

In [20]:
from src.silver.cdc_reconc_historical_load import seed as seed_reconciliation
from src.silver.enterprise_company import seed as seed_enterprise_company

seed_reconciliation(
    runs_parquet=DATA / 'reconciliation_runs.parquet',
    results_parquet=DATA / 'reconciliation_results.parquet',
    force=True,
)
seed_enterprise_company(DATA / 'enterprise_company.parquet', force=True)

WARNING  --force: dropping and recreating silver reconciliation tables.
INFO  Seed complete: 13 runs, 1000 results. seq_run_id starts at 14, seq_result_id at 1001.
WARNING  --force: truncating silver_enterprise_company.
INFO  Loaded 10 merchants into silver_enterprise_company.


10

## 3 — Row counts per table

In [21]:
import pandas as pd
from src.db import get_connection

conn = get_connection()

tables = [
    'raw_transactions',
    'silver_reconciliation_runs',
    'silver_reconciliation_results',
    'silver_enterprise_company',
]

rows = []
for t in tables:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    rows.append({'table': t, 'rows': n})

counts_df = pd.DataFrame(rows)
display(counts_df)

for r in rows:
    assert r['rows'] > 0, f"Table {r['table']} is empty — expected rows after load"

,table,rows
0,raw_transactions,955
1,silver_reconciliation_runs,13
2,silver_reconciliation_results,1000
3,silver_enterprise_company,10


## 4 — CDC: no deleted rows should survive in `raw_transactions`

Silver tables have CDC applied at seed time, so only `raw_transactions` needs this check.

In [22]:
deleted = conn.execute("SELECT COUNT(*) FROM raw_transactions WHERE Op = 'D'").fetchone()[0]
status = 'OK' if deleted == 0 else 'FAIL'
print(f'[{status}] raw_transactions: {deleted} deleted rows remaining')
assert deleted == 0, f"{deleted} row(s) with Op='D' survived CDC filtering"

[OK] raw_transactions: 0 deleted rows remaining


## 5 — CDC: each PK appears exactly once (deduplication check)

In [23]:
pk_map = {
    'raw_transactions':              'transaction_id',
    'silver_reconciliation_runs':    'id',
    'silver_reconciliation_results': 'id',
    'silver_enterprise_company':     'id',
}

for t, pk in pk_map.items():
    dups = conn.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT {pk}, COUNT(*) AS cnt FROM {t} GROUP BY {pk} HAVING cnt > 1
        )
    """).fetchone()[0]
    status = 'OK' if dups == 0 else 'FAIL'
    print(f'[{status}] {t}: {dups} duplicate {pk}s')
    assert dups == 0, f"{t} has {dups} duplicate {pk} value(s) after CDC dedup"

[OK] raw_transactions: 0 duplicate transaction_ids
[OK] silver_reconciliation_runs: 0 duplicate ids
[OK] silver_reconciliation_results: 0 duplicate ids
[OK] silver_enterprise_company: 0 duplicate ids


## 6 — NULL _timestamp check

The loader now validates this before running the dedup window. Confirm raw_transactions has none.

In [24]:
null_ts = conn.execute(
    "SELECT COUNT(*) FROM raw_transactions WHERE _timestamp IS NULL"
).fetchone()[0]
status = 'OK' if null_ts == 0 else 'FAIL'
print(f'[{status}] raw_transactions: {null_ts} NULL _timestamp rows')
assert null_ts == 0, f"{null_ts} row(s) have NULL _timestamp in raw_transactions"

[OK] raw_transactions: 0 NULL _timestamp rows


## 7 — Schema drift: `payment_method` merged from batch_2 into raw_transactions

In [25]:
import pyarrow.parquet as pq

batch1_rows = pq.read_metadata(DATA / 'transactions_batch_1.parquet').num_rows
print(f'batch_1 parquet row count: {batch1_rows}')

cols = [r[0] for r in conn.execute('DESCRIBE raw_transactions').fetchall()]
print('Columns:', cols)
assert 'payment_method' in cols, "payment_method column missing — schema drift merge failed"
print('[OK] payment_method column present')

drift_df = conn.execute(f"""
    SELECT
        CASE WHEN id <= {batch1_rows} THEN 'batch_1_origin' ELSE 'batch_2_origin' END AS origin,
        COUNT(*)                                                                        AS total,
        COUNT(payment_method)                                                           AS with_payment_method
    FROM raw_transactions
    GROUP BY origin
    ORDER BY origin
""").df()
display(drift_df)

# After CDC dedup, batch_1-origin rows may have payment_method if batch_2 had an update
# for the same transaction_id — the winning row carries the batch_2 value. Just report it.
for _, row in drift_df.iterrows():
    total = int(row['total'])
    with_pm = int(row['with_payment_method'])
    fill_rate = with_pm / total if total > 0 else 0
    origin = row['origin']
    print(f'[INFO] {origin}: payment_method fill rate {fill_rate:.1%} ({with_pm}/{total})')

b2 = drift_df[drift_df['origin'] == 'batch_2_origin']
if not b2.empty:
    b2_total = int(b2['total'].iloc[0])
    b2_with = int(b2['with_payment_method'].iloc[0])
    fill_rate = b2_with / b2_total if b2_total > 0 else 0
    if fill_rate >= 0.9:
        print(f'[OK] batch_2_origin payment_method fill rate: {fill_rate:.1%}')
    else:
        print(f'[WARNING] batch_2_origin payment_method fill rate is low: {fill_rate:.1%} ({b2_with}/{b2_total})')

batch_1 parquet row count: 593
Columns: ['id', 'transaction_id', 'merchant_id', 'amount', 'currency', 'status', 'description', 'created_at', 'updated_at', 'Op', '_timestamp', 'payment_method']
[OK] payment_method column present


,origin,total,with_payment_method
0,batch_1_origin,570,118
1,batch_2_origin,385,106


[INFO] batch_1_origin: payment_method fill rate 20.7% (118/570)
[INFO] batch_2_origin: payment_method fill rate 27.5% (106/385)
[WARNING] batch_2_origin payment_method fill rate is low: 27.5% (106/385)


## 8 — Idempotency: running twice must not change row counts

`load_transactions` uses `CREATE OR REPLACE`, so it's always idempotent.  
Silver `seed()` raises if the table already has rows — pass `force=True` to re-seed.

In [26]:
before = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tables}

load_transactions([
    DATA / 'transactions_batch_1.parquet',
    DATA / 'transactions_batch_2.parquet',
])
seed_reconciliation(
    runs_parquet=DATA / 'reconciliation_runs.parquet',
    results_parquet=DATA / 'reconciliation_results.parquet',
    force=True,
)
seed_enterprise_company(DATA / 'enterprise_company.parquet', force=True)

after = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tables}

for t in tables:
    status = 'OK' if before[t] == after[t] else 'FAIL'
    print(f'[{status}] {t}: {before[t]} → {after[t]}')
    assert before[t] == after[t], f"{t} row count changed on re-run: {before[t]} → {after[t]}"

INFO  raw_transactions: 956 raw rows → 955 final rows (1 removed by dedup+deletes)


WARNING  --force: dropping and recreating silver reconciliation tables.
INFO  Seed complete: 13 runs, 1000 results. seq_run_id starts at 14, seq_result_id at 1001.
WARNING  --force: truncating silver_enterprise_company.
INFO  Loaded 10 merchants into silver_enterprise_company.


[OK] raw_transactions: 955 → 955
[OK] silver_reconciliation_runs: 13 → 13
[OK] silver_reconciliation_results: 1000 → 1000
[OK] silver_enterprise_company: 10 → 10


## 9 — raw_transactions sample

In [27]:
conn.execute("SELECT * FROM raw_transactions ORDER BY _timestamp DESC LIMIT 10").df()

,id,transaction_id,merchant_id,amount,currency,status,description,created_at,updated_at,Op,_timestamp,payment_method
0,186,3d877034-7217-4fa3-9893-ef12ab3f8d22,MERCH_0010,27.16,BRL,COMPLETED,Payment 186,2025-03-05T23:38:25+00:00,2025-03-06T00:43:25+00:00,I,2025-03-05T23:38:25+00:00,debit_card
1,806,e4cda3b9-ac97-4067-88bb-ec177ccd3eb6,MERCH_0009,67.85,BRL,COMPLETED,Payment 806,2025-03-05T23:23:11+00:00,2025-03-06T01:51:11+00:00,I,2025-03-05T23:23:11+00:00,NaN
2,604,2d8b62b6-89cb-49aa-8f56-fffcc1bbb767,MERCH_0007,210.35,BRL,COMPLETED,Payment 604,2025-03-05T23:23:08+00:00,2025-03-06T00:53:08+00:00,I,2025-03-05T23:23:08+00:00,boleto
3,993,0faae5bf-b900-4086-aa21-e1537b2cefe1,MERCH_0004,57.29,BRL,COMPLETED,Payment 993,2025-03-05T23:20:45+00:00,2025-03-06T00:43:45+00:00,I,2025-03-05T23:20:45+00:00,NaN
4,793,4eea155b-c96e-417f-a49e-6f6e6c561f78,MERCH_0010,56.91,BRL,COMPLETED,Payment 793,2025-03-05T23:17:44+00:00,2025-03-06T00:20:44+00:00,I,2025-03-05T23:17:44+00:00,pix
5,706,1f16bfe5-df8b-4c1c-a7d3-584cfb99ac3e,MERCH_0001,901.42,USD,COMPLETED,Payment 706,2025-03-05T23:15:57+00:00,2025-03-06T00:00:57+00:00,I,2025-03-05T23:15:57+00:00,boleto
6,498,1b8fabe9-5237-4d88-ac8f-abbcec6dd411,MERCH_0008,108.83,BRL,FAILED,Payment 498,2025-03-05T22:59:04+00:00,2025-03-06T00:51:04+00:00,I,2025-03-05T22:59:04+00:00,NaN
7,943,77cf865f-71fe-4889-9af9-469ad55957c2,MERCH_0010,111.71,BRL,COMPLETED,Payment 943,2025-03-05T22:56:51+00:00,2025-03-05T23:49:51+00:00,I,2025-03-05T22:56:51+00:00,debit_card
8,665,7d84dae5-af52-45bc-b8ab-33f4f2f9ade1,MERCH_0001,161.42,BRL,PENDING,Payment 665,2025-03-05T22:34:55+00:00,2025-03-05T22:57:55+00:00,I,2025-03-05T22:34:55+00:00,pix
9,906,99222b37-2d9e-42d2-9ade-a2a4b1c72493,MERCH_0006,28.10,BRL,COMPLETED,Payment 906,2025-03-05T22:33:09+00:00,2025-03-05T22:38:09+00:00,I,2025-03-05T22:33:09+00:00,debit_card


## 10 — silver_enterprise_company sample (with updates applied)

In [28]:
conn.execute("SELECT * FROM silver_enterprise_company ORDER BY id LIMIT 10").df()

,id,merchant_id,legal_name,trade_name,document,primary_cnae,created_at,updated_at
0,1,MERCH_0001,Empresa 0001 LTDA,Loja 0001,44463998825047,5611203,2024-11-23 01:00:00+01:00,2024-12-21 01:00:00+01:00
1,2,MERCH_0002,Empresa 0002 LTDA,Loja 0002,86754897461584,4712100,2024-03-12 01:00:00+01:00,2024-04-07 02:00:00+02:00
2,3,MERCH_0003,Empresa 0003 LTDA,Loja 0003,14192024586483,4712100,2024-10-29 01:00:00+01:00,2025-02-14 01:00:00+01:00
3,4,MERCH_0004,Empresa 0004 LTDA,Loja 0004,94724695365329,4712100,2024-04-21 02:00:00+02:00,2024-06-19 02:00:00+02:00
4,5,MERCH_0005,Empresa 0005 LTDA,Loja 0005,86693948205196,6201501,2024-10-14 02:00:00+02:00,2024-12-03 01:00:00+01:00
5,6,MERCH_0006,Empresa 0006 LTDA,Loja 0006,49151157779884,4712100,2024-04-22 02:00:00+02:00,2024-08-14 02:00:00+02:00
6,7,MERCH_0007,Empresa 0007 LTDA,Loja 0007,57886405498129,4711302,2024-03-22 01:00:00+01:00,2024-09-16 02:00:00+02:00
7,8,MERCH_0008,Empresa 0008 LTDA,Loja 0008,24385291136889,4712100,2024-03-20 01:00:00+01:00,2024-05-14 02:00:00+02:00
8,9,MERCH_0009,Empresa 0009 LTDA,Loja 0009,94964520328049,4711302,2024-07-13 02:00:00+02:00,2024-08-06 02:00:00+02:00
9,10,MERCH_0010,Empresa 0010 LTDA,Loja 0010,27568719322757,6201501,2024-01-23 01:00:00+01:00,2024-05-19 02:00:00+02:00


In [12]:
conn.close()